In [1]:
%load_ext autoreload
%autoreload 2
    
import sys
import os

# Aggiunge la cartella 'src' relativa alla posizione dello script
sys.path.append(os.path.abspath("src"))

## Import ISO14224 to NEO4J

In [2]:
from iso2neo4j import PlantDataImporter
graph_importer = PlantDataImporter("neo4j://neo4j","neo4j","password")
graph_importer.clean_all_data()
graph_importer.import_plant_data("../data/acme-plant.json")

## Import data to InfluxDB

In [3]:
# get list of all equipments
equipments = graph_importer.get_equipments()

In [4]:
equipments 

,EQUIPMENT
0,ACME Limited.ACME Plant.PUMP-001
1,ACME Limited.ACME Plant.PUMP-002
2,ACME Limited.ACME Plant.WIND-TURB-01


In [5]:
from csv2influxdb import EquipmentCsvInfluxImporter
data_importer = EquipmentCsvInfluxImporter("http://influxdb:8086", "MyInitialAdminToken0==", "iiot", "measures", "../data")
data_importer.import_all(equipments['EQUIPMENT'].values)

/home/jovyan/work/src/csv2influxdb.py:106: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[c] = pd.to_numeric(df[c], errors="ignore")


                  timestamp  Vibration Velocity  sensor_01  \
0 2025-03-30 00:00:00+00:00            2.465394   47.09201   
1 2025-03-30 00:01:00+00:00            2.465394   47.09201   
2 2025-03-30 00:02:00+00:00            2.444734   47.35243   
3 2025-03-30 00:03:00+00:00            2.460474   47.09201   
4 2025-03-30 00:04:00+00:00            2.445718   47.13541   

   Bearing Temperature  sensor_03  sensor_04  sensor_05  Discharge Pressure  \
0              53.2118  46.310760   634.3750   76.45975            13.41146   
1              53.2118  46.310760   634.3750   76.45975            13.41146   
2              53.2118  46.397570   638.8889   73.54598            13.32465   
3              53.1684  46.397568   628.1250   76.98898            13.31742   
4              53.2118  46.397568   636.4583   76.58897            13.35359   

   sensor_07  
0   16.13136  
1   16.13136  
2   16.03733  
3   16.24711  
4   16.21094  


/home/jovyan/work/src/csv2influxdb.py:106: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[c] = pd.to_numeric(df[c], errors="ignore")


                  timestamp  LV ActivePower  Wind Speed  \
0 2024-12-30 00:00:00+00:00      380.047791    5.311336   
1 2024-12-30 00:10:00+00:00      453.769196    5.672167   
2 2024-12-30 00:20:00+00:00      306.376587    5.216037   
3 2024-12-30 00:30:00+00:00      419.645905    5.659674   
4 2024-12-30 00:40:00+00:00      380.650696    5.577941   

   Theoretical_Power_Curve  Wind Direction  
0               416.328908      259.994904  
1               519.917511      268.641113  
2               390.900016      272.564789  
3               516.127569      271.258087  
4               491.702972      265.674286  


{'ACME Limited.ACME Plant.PUMP-001': 'imported',
 'ACME Limited.ACME Plant.PUMP-002': 'missing',
 'ACME Limited.ACME Plant.WIND-TURB-01': 'imported'}

## Sync the value among the 2 DBs
* foreach mesure get last valu and write to neo4j
* you can schedule this sync using AirfLow or Kubernetes sheduler see [Hands-On Industrial Internet of Things: Build robust industrial IoT infrastructure by using the cloud and artificial intelligence 2nd ed. Edition](https://a.co/d/0dfb4HKH)

In [6]:
from syncdbs import SyncDBs
sync = SyncDBs(data_importer.client, graph_importer.driver)


In [7]:
sync.sync()

ACME Limited.ACME Plant.PUMP-001 Bearing Temperature updated to value= 56.032990000000005
ACME Limited.ACME Plant.PUMP-001 Discharge Pressure updated to value= 22.25116
ACME Limited.ACME Plant.PUMP-001 Vibration Velocity updated to value= 2.549016
Empy result ACME Limited.ACME Plant.PUMP-002 Motor Current []
Empy result ACME Limited.ACME Plant.PUMP-002 Suction Pressure []
Empy result ACME Limited.ACME Plant.PUMP-002 Vibration Velocity []
ACME Limited.ACME Plant.WIND-TURB-01 LV ActivePower updated to value= 3618.73291015625
ACME Limited.ACME Plant.WIND-TURB-01 Wind Direction updated to value= 359.997589111328
ACME Limited.ACME Plant.WIND-TURB-01 Wind Speed updated to value= 25.2060108184814


In [8]:
data_importer.close()
graph_importer.close()

# Graph Rag

In [9]:
from graphrag_app import OllamaGraphRAG
app = OllamaGraphRAG("neo4j://neo4j","neo4j","password")
app.chat_with_rag("Which events occurred at Equipment PUMP-001?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:

MATCH (c:Company)-[:HAS_INSTALLATION]->(i:Installation)-[:HAS_EQUIPMENT]->(e:Equipment {id: 'PUMP-001'})-[:EXPERIENCED_FAILURE]->(f:FailureEvent)
RETURN f.id, f.impact, f.down_time_hrs, f.mode, f.mechanism

Full Context:
[{'f.id': 'FAIL-P001-01', 'f.impact': 'Critical', 'f.down_time_hrs': 14.0, 'f.mode': 'External leakage', 'f.mechanism': 'Wear'}]

> Finished chain.


{'query': 'Which events occurred at Equipment PUMP-001?',
 'result': ' The event recorded at Equipment PUMP-001 was a Critical external leakage due to wear, which lasted for 14 hours.'}

In [12]:
app.chat_with_rag("What is the Wind Speed of Equipment WIND-TURB-01?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:

MATCH (c:Company)-[:HAS_INSTALLATION]->(i:Installation)-[:HAS_EQUIPMENT]->(e:Equipment)
WHERE e.id = 'WIND-TURB-01'
OPTIONAL MATCH (e)-[:HAS_MEASUREMENT]->(m:Measurement)
WHERE m.name = 'Wind Speed'
RETURN m.value

Full Context:
[{'m.value': 25.2060108184814}]

> Finished chain.


{'query': 'What is the Wind Speed of Equipment WIND-TURB-01?',
 'result': ' The wind speed of Equipment WIND-TURB-01 is 25.2060108184814 [m/s].'}

In [11]:
app.chat_with_rag("Can you get me the High critical equipment of Installation 'ACME Plant'?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:

MATCH (c:Company)-[:HAS_INSTALLATION]->(i:Installation {name: 'ACME Plant'})-[:HAS_EQUIPMENT]->(e:Equipment)
WHERE e.criticality = 'High'
RETURN e

Full Context:
[{'e': {'criticality': 'High', 'model': 'X-Flow 500', 'operating_mode': 'Continuous', 'id': 'PUMP-001', 'manufacturer': 'FluidDrive Corp'}}]

> Finished chain.


{'query': "Can you get me the High critical equipment of Installation 'ACME Plant'?",
 'result': " The High critical equipment of Installation 'ACME Plant' is the X-Flow 500, which is manufactured by FluidDrive Corp and operates in Continuous mode."}